# SAM3 Training Setup - Practical Implementation

> Step-by-step implementation guide to get started with SAM3 fine-tuning on custom datasets

This notebook walks through the actual implementation of training setup, not just documentation.

## Prerequisites Check

First, let's check what we have and what we need.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import json
import yaml
from typing import Dict, Any

# Define paths
WORKSPACE = Path("/workspace").resolve() if Path("/workspace").exists() else Path.cwd()
SAM3_REPO = WORKSPACE / "sam3"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "nbs" else Path.cwd()
CONFIG_DIR = PROJECT_ROOT / "configs"
TRAINING_DIR = PROJECT_ROOT / "training"
DATA_ROOT = WORKSPACE / "data" / "autolabel"

print(f"Workspace: {WORKSPACE}")
print(f"SAM3 Repo Path: {SAM3_REPO}")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Config Directory: {CONFIG_DIR}")
print(f"Training Directory: {TRAINING_DIR}")
print(f"Data Root: {DATA_ROOT}")

## Step 1: Clone and Setup SAM3 Repository

Let's clone the official SAM3 repository if it doesn't exist.

In [ ]:
def clone_sam3_repo(target_path: Path) -> bool:
    """Clone SAM3 repository if it doesn't exist."""
    if target_path.exists():
        print(f"✓ SAM3 repository already exists at {target_path}")
        return True
    
    print(f"Cloning SAM3 repository to {target_path}...")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        subprocess.run(
            ["git", "clone", "https://github.com/facebookresearch/sam3.git", str(target_path)],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Successfully cloned SAM3 repository")
        return True
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed to clone repository: {e.stderr}")
        return False

clone_sam3_repo(SAM3_REPO)

## Step 2: Install SAM3 Dependencies

Install SAM3 with training dependencies.

In [ ]:
def install_sam3_training_deps(repo_path: Path) -> bool:
    """Install SAM3 with training dependencies."""
    if not repo_path.exists():
        print(f"✗ SAM3 repository not found at {repo_path}")
        return False
    
    print("Installing SAM3 with training dependencies...")
    try:
        # Install in editable mode with train and dev extras
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", ".[train,dev]"],
            cwd=str(repo_path),
            check=True,
            capture_output=True,
            text=True
        )
        print("✓ Successfully installed SAM3 with training dependencies")
        return True
    except subprocess.CalledProcessError as e:
        print(f"✗ Installation failed: {e.stderr}")
        print("You may need to install manually:")
        print(f"  cd {repo_path}")
        print(f"  pip install -e .[train,dev]")
        return False

# Uncomment to install (may take a few minutes)
# install_sam3_training_deps(SAM3_REPO)

## Step 3: Verify Data Preparation

Check if we have auto-labeled data from the previous pipeline.

In [ ]:
def verify_training_data(data_root: Path) -> Dict[str, Any]:
    """Verify training data exists and is properly formatted."""
    status = {
        "exists": False,
        "images": 0,
        "masks": 0,
        "annotations": False,
        "annotation_path": None
    }
    
    if not data_root.exists():
        print(f"✗ Data directory not found: {data_root}")
        print("  Run the 06_sam3_autolabel_pipeline.ipynb notebook first to generate training data.")
        return status
    
    status["exists"] = True
    
    # Check images
    images_dir = data_root / "images"
    if images_dir.exists():
        status["images"] = len(list(images_dir.glob("*.png")))
    
    # Check masks
    masks_dir = data_root / "masks"
    if masks_dir.exists():
        status["masks"] = len(list(masks_dir.glob("*.png")))
    
    # Check annotations
    annotation_file = data_root / "autolabel_annotations.json"
    if annotation_file.exists():
        status["annotations"] = True
        status["annotation_path"] = annotation_file
    
    # Print status
    print(f"Data Directory Status:")
    print(f"  ✓ Directory exists: {data_root}")
    print(f"  {'✓' if status['images'] > 0 else '✗'} Images: {status['images']}")
    print(f"  {'✓' if status['masks'] > 0 else '✗'} Masks: {status['masks']}")
    print(f"  {'✓' if status['annotations'] else '✗'} Annotations: {annotation_file}")
    
    if status["images"] == 0:
        print("\n⚠️  No training data found. Please run 06_sam3_autolabel_pipeline.ipynb first.")
    
    return status

data_status = verify_training_data(DATA_ROOT)
data_status

## Step 4: Create Training Configuration File

Generate a YAML configuration file for fine-tuning SAM3.

In [ ]:
def create_training_config(
    output_path: Path,
    data_root: Path,
    checkpoint_path: str = "facebook/sam3",
    batch_size: int = 4,
    num_epochs: int = 10,
    learning_rate: float = 8e-5,
    num_gpus: int = 1
) -> Path:
    """Create a training configuration file."""
    
    config = {
        "experiment_name": "sam3_custom_finetune",
        
        # Model configuration
        "model": {
            "checkpoint_path": checkpoint_path,
            "load_from_HF": True,
            "enable_segmentation": True,
            "enable_inst_interactivity": False,
            "eval_mode": False
        },
        
        # Dataset configuration
        "dataset": {
            "name": "custom_dataset",
            "root": str(data_root),
            "annotation_file": str(data_root / "autolabel_annotations.json"),
            "image_dir": str(data_root / "images"),
            "format": "coco",
            "splits": {
                "train": "train",
                "val": "val"
            }
        },
        
        # Training configuration
        "training": {
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "learning_rate": learning_rate,
            "optimizer": "AdamW",
            "weight_decay": 0.1,
            "gradient_clip": 0.1,
            
            # Learning rate schedule
            "scheduler": {
                "type": "InverseSquareRoot",
                "warmup_steps": 20,
                "timescale": 20,
                "cooldown_steps": 20
            },
            
            # Component-specific learning rates
            "lr_scale": {
                "transformer": 1.0,
                "vision_backbone": 0.3125,  # 2.5e-5 / 8e-5
                "language_backbone": 0.0625  # 5e-6 / 8e-5
            },
            
            # Data augmentation
            "augmentation": {
                "random_flip": True,
                "random_crop": False,
                "color_jitter": False
            },
            
            # Resolution
            "resolution": 1008,
            "max_annotations_per_image": 200
        },
        
        # Distributed training configuration
        "launcher": {
            "use_cluster": False,
            "num_gpus": num_gpus,
            "num_nodes": 1,
            "distributed_backend": "nccl"
        },
        
        # Logging and checkpointing
        "logging": {
            "output_dir": "./outputs/sam3_finetune",
            "log_frequency": 10,
            "save_checkpoint_frequency": 500,
            "eval_frequency": 100,
            "wandb": {
                "enabled": False,
                "project": "sam3-finetune",
                "entity": None
            }
        },
        
        # Scratch training mode (for segmentation)
        "scratch": {
            "enable_segmentation": True,
            "segmentation_loss_weight": 1.0,
            "detection_loss_weight": 1.0
        }
    }
    
    # Ensure output directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Write config to YAML file
    with open(output_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    print(f"✓ Created training configuration: {output_path}")
    return output_path

# Create config directory
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Create training configuration
config_path = create_training_config(
    output_path=CONFIG_DIR / "sam3_custom_finetune.yaml",
    data_root=DATA_ROOT,
    batch_size=4,
    num_epochs=10,
    learning_rate=8e-5,
    num_gpus=1
)

# Display the config
with open(config_path, 'r') as f:
    print("\nGenerated Configuration:")
    print("=" * 50)
    print(f.read())

## Step 5: Create Training Launcher Script

Create a Python script that can launch the training.

In [ ]:
def create_training_launcher(output_path: Path, sam3_repo: Path, config_path: Path) -> Path:
    """Create a training launcher script."""
    
    script_content = f'''#!/usr/bin/env python3
"""
SAM3 Training Launcher Script

This script launches SAM3 fine-tuning with the specified configuration.

Usage:
    python train_sam3.py [--config CONFIG_PATH] [--num-gpus NUM_GPUS]

Example:
    python train_sam3.py --config configs/sam3_custom_finetune.yaml --num-gpus 1
"""

import argparse
import os
import sys
import subprocess
from pathlib import Path

# Add SAM3 to path
SAM3_REPO = Path("{sam3_repo}").resolve()
if SAM3_REPO.exists():
    sys.path.insert(0, str(SAM3_REPO))
else:
    print(f"Error: SAM3 repository not found at {{SAM3_REPO}}")
    print("Please clone the repository first:")
    print("  git clone https://github.com/facebookresearch/sam3.git /workspace/sam3")
    sys.exit(1)

def parse_args():
    parser = argparse.ArgumentParser(description="Launch SAM3 Training")
    parser.add_argument(
        "--config",
        type=str,
        default="{config_path}",
        help="Path to training configuration file"
    )
    parser.add_argument(
        "--num-gpus",
        type=int,
        default=1,
        help="Number of GPUs to use"
    )
    parser.add_argument(
        "--use-cluster",
        type=int,
        default=0,
        help="Whether to use cluster (SLURM) for training"
    )
    parser.add_argument(
        "--mode",
        type=str,
        default="train",
        choices=["train", "val"],
        help="Training or validation mode"
    )
    return parser.parse_args()

def check_prerequisites():
    """Check if all prerequisites are met."""
    print("Checking prerequisites...")
    
    # Check SAM3 installation
    try:
        import sam3
        print("✓ SAM3 is installed")
    except ImportError:
        print("✗ SAM3 is not installed")
        print("  Please install: pip install -e /workspace/sam3[train,dev]")
        return False
    
    # Check CUDA availability
    try:
        import torch
        if torch.cuda.is_available():
            print(f"✓ CUDA available: {{torch.cuda.device_count()}} GPU(s)")
        else:
            print("⚠️  CUDA not available, will use CPU (very slow)")
    except ImportError:
        print("✗ PyTorch not installed")
        return False
    
    return True

def launch_training(config_path: str, num_gpus: int, use_cluster: int, mode: str):
    """Launch SAM3 training."""
    train_script = SAM3_REPO / "sam3" / "train" / "train.py"
    
    if not train_script.exists():
        print(f"Error: Training script not found at {{train_script}}")
        print("Please ensure the SAM3 repository is properly cloned.")
        return False
    
    # Build command
    cmd = [
        sys.executable,
        str(train_script),
        "-c", config_path,
        "--use-cluster", str(use_cluster),
        "--num-gpus", str(num_gpus),
    ]
    
    if mode == "val":
        cmd.extend(["trainer.mode=val"])
    
    print(f"\nLaunching training with command:")
    print(" ".join(cmd))
    print("\n" + "="*70 + "\n")
    
    # Launch training
    try:
        subprocess.run(cmd, check=True)
        print("\n" + "="*70)
        print("✓ Training completed successfully")
        return True
    except subprocess.CalledProcessError as e:
        print(f"\n✗ Training failed with error code {{e.returncode}}")
        return False
    except KeyboardInterrupt:
        print("\n⚠️  Training interrupted by user")
        return False

def main():
    args = parse_args()
    
    print("SAM3 Training Launcher")
    print("="*70)
    print(f"Configuration: {{args.config}}")
    print(f"GPUs: {{args.num_gpus}}")
    print(f"Mode: {{args.mode}}")
    print("="*70 + "\n")
    
    # Check prerequisites
    if not check_prerequisites():
        print("\nPrerequisites not met. Please fix the issues above.")
        sys.exit(1)
    
    # Launch training
    success = launch_training(
        config_path=args.config,
        num_gpus=args.num_gpus,
        use_cluster=args.use_cluster,
        mode=args.mode
    )
    
    sys.exit(0 if success else 1)

if __name__ == "__main__":
    main()
'''
    
    # Write script
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        f.write(script_content)
    
    # Make executable
    output_path.chmod(0o755)
    
    print(f"✓ Created training launcher: {output_path}")
    return output_path

# Create training directory
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

# Create launcher script
launcher_path = create_training_launcher(
    output_path=TRAINING_DIR / "train_sam3.py",
    sam3_repo=SAM3_REPO,
    config_path=config_path
)

print(f"\nTraining launcher created at: {launcher_path}")
print(f"\nTo start training, run:")
print(f"  python {launcher_path} --config {config_path} --num-gpus 1")

## Step 6: Create Quick Start README

Create a README file with quick start instructions.

In [ ]:
def create_training_readme(output_path: Path, config_path: Path, launcher_path: Path) -> Path:
    """Create a training README file."""
    
    readme_content = f'''# SAM3 Fine-Tuning Quick Start Guide

This directory contains everything you need to fine-tune SAM3 on your custom dataset.

## Prerequisites

1. **SAM3 Repository**: Clone the official repository
   ```bash
   cd /workspace
   git clone https://github.com/facebookresearch/sam3.git
   cd sam3
   pip install -e ".[train,dev]"
   ```

2. **Model Checkpoint**: Request access and download from HuggingFace
   - Visit: https://huggingface.co/facebook/sam3
   - Request access to the model
   - Login: `huggingface-cli login`

3. **Training Data**: Generate using the auto-label pipeline
   ```bash
   # Run notebook: nbs/06_sam3_autolabel_pipeline.ipynb
   # Set environment: RUN_SAM3=1
   ```

## Quick Start

### 1. Verify Your Setup

```bash
# Check if SAM3 is installed
python -c "import sam3; print('SAM3 installed successfully')"

# Check GPU availability
python -c "import torch; print(f'CUDA available: {{torch.cuda.is_available()}}')"

# Check training data
ls -la /workspace/data/autolabel/
```

### 2. Review Configuration

The training configuration is located at:
```
{config_path}
```

Key parameters you can modify:
- `batch_size`: Adjust based on your GPU memory (default: 4)
- `num_epochs`: Number of training epochs (default: 10)
- `learning_rate`: Base learning rate (default: 8e-5)
- `num_gpus`: Number of GPUs to use (default: 1)

### 3. Launch Training

**Single GPU:**
```bash
python {launcher_path} \\
    --config {config_path} \\
    --num-gpus 1
```

**Multi-GPU (4 GPUs):**
```bash
python {launcher_path} \\
    --config {config_path} \\
    --num-gpus 4
```

**Evaluation Mode:**
```bash
python {launcher_path} \\
    --config {config_path} \\
    --num-gpus 1 \\
    --mode val
```

### 4. Monitor Training

Training outputs will be saved to:
```
./outputs/sam3_finetune/
├── checkpoints/
├── logs/
└── tensorboard/
```

View training logs:
```bash
tail -f outputs/sam3_finetune/logs/train.log
```

Launch TensorBoard:
```bash
tensorboard --logdir outputs/sam3_finetune/tensorboard
```

## Configuration Details

### Model Configuration
- Checkpoint: `facebook/sam3` (from HuggingFace)
- Segmentation enabled: Yes
- Interactive mode: No

### Training Hyperparameters
- **Optimizer**: AdamW
- **Learning Rate**: 8e-5 (transformer), 2.5e-5 (vision), 5e-6 (language)
- **Weight Decay**: 0.1
- **Gradient Clipping**: 0.1
- **Scheduler**: InverseSquareRoot with warmup
- **Warmup Steps**: 20
- **Resolution**: 1008

### Dataset
- Format: COCO
- Images: `/workspace/data/autolabel/images/`
- Annotations: `/workspace/data/autolabel/autolabel_annotations.json`

## Troubleshooting

### CUDA Out of Memory
- Reduce `batch_size` in config (try 2 or 1)
- Reduce `resolution` (try 512)
- Use gradient accumulation

### HuggingFace Access Error
- Ensure you've requested access to `facebook/sam3`
- Login: `huggingface-cli login`
- Wait for approval (usually within 24 hours)

### Import Error: No module named 'sam3'
- Install SAM3: `cd /workspace/sam3 && pip install -e ".[train,dev]"`
- Check installation: `pip show sam3`

### No Training Data
- Run the auto-label pipeline: `nbs/06_sam3_autolabel_pipeline.ipynb`
- Set `RUN_SAM3=1` to use real SAM3 model
- Verify data: `ls -la /workspace/data/autolabel/`

## Advanced Options

### Custom Dataset
Modify the config to point to your custom dataset:
```yaml
dataset:
  root: "/path/to/your/data"
  annotation_file: "/path/to/your/annotations.json"
  image_dir: "/path/to/your/images"
```

### Resume Training
Add to config:
```yaml
training:
  resume_from: "./outputs/sam3_finetune/checkpoints/checkpoint_1000.pth"
```

### Enable Weights & Biases
Modify config:
```yaml
logging:
  wandb:
    enabled: true
    project: "sam3-finetune"
    entity: "your-username"
```

## Next Steps

After training completes:

1. **Evaluate** your model on validation data
2. **Export** the best checkpoint
3. **Run inference** on new images
4. **Iterate** by adjusting hyperparameters

## Resources

- SAM3 GitHub: https://github.com/facebookresearch/sam3
- SAM3 Paper: https://arxiv.org/abs/2511.16719
- HuggingFace: https://huggingface.co/facebook/sam3
- Documentation: See `07_sam3_training_documentation.ipynb`
'''
    
    # Write README
    with open(output_path, 'w') as f:
        f.write(readme_content)
    
    print(f"✓ Created training README: {output_path}")
    return output_path

# Create README
readme_path = create_training_readme(
    output_path=TRAINING_DIR / "README.md",
    config_path=config_path,
    launcher_path=launcher_path
)

print(f"\n✓ Training setup complete!")
print(f"\nCreated files:")
print(f"  - Configuration: {config_path}")
print(f"  - Launcher: {launcher_path}")
print(f"  - README: {readme_path}")
print(f"\nRead {readme_path} for instructions.")

## Step 7: Test Training Setup (Dry Run)

Let's verify that everything is configured correctly without actually running training.

In [ ]:
def test_training_setup():
    """Run checks to verify training setup is correct."""
    print("Running Training Setup Validation...")
    print("=" * 70)
    
    checks = {
        "SAM3 Repository": SAM3_REPO.exists(),
        "Configuration File": config_path.exists(),
        "Launcher Script": launcher_path.exists(),
        "Training README": readme_path.exists(),
        "Data Directory": DATA_ROOT.exists(),
        "Images": (DATA_ROOT / "images").exists(),
        "Annotations": (DATA_ROOT / "autolabel_annotations.json").exists()
    }
    
    all_passed = True
    for check_name, passed in checks.items():
        status = "✓" if passed else "✗"
        print(f"{status} {check_name}")
        if not passed:
            all_passed = False
    
    print("=" * 70)
    
    if all_passed:
        print("\n🎉 All checks passed! You're ready to start training.")
        print(f"\nTo begin training, run:")
        print(f"  python {launcher_path} --config {config_path} --num-gpus 1")
    else:
        print("\n⚠️  Some checks failed. Please address the issues above.")
        if not data_status["images"]:
            print("\nMissing training data! Run: nbs/06_sam3_autolabel_pipeline.ipynb")
        if not SAM3_REPO.exists():
            print("\nMissing SAM3 repository! Clone it first.")
    
    return all_passed

test_training_setup()

## Summary

This notebook has created a complete training setup:

### ✅ What Was Created

1. **Training Configuration** (`configs/sam3_custom_finetune.yaml`)
   - Complete YAML config with all hyperparameters
   - Optimized for fine-tuning on custom datasets
   - Component-specific learning rates configured

2. **Training Launcher** (`training/train_sam3.py`)
   - Ready-to-use Python script
   - Handles prerequisites checking
   - Supports single/multi-GPU training
   - Command-line interface

3. **Quick Start Guide** (`training/README.md`)
   - Step-by-step instructions
   - Troubleshooting section
   - Configuration details
   - Advanced options

### 📋 Next Actions

1. **Clone SAM3**: `git clone https://github.com/facebookresearch/sam3.git /workspace/sam3`
2. **Install Dependencies**: `cd /workspace/sam3 && pip install -e ".[train,dev]"`
3. **Get Checkpoint**: Request access at https://huggingface.co/facebook/sam3
4. **Generate Data**: Run `06_sam3_autolabel_pipeline.ipynb` with `RUN_SAM3=1`
5. **Start Training**: `python training/train_sam3.py --config configs/sam3_custom_finetune.yaml --num-gpus 1`

### 🎯 Key Parameters in Config

- **Optimizer**: AdamW
- **Learning Rates**: 8e-5 (transformer), 2.5e-5 (vision), 5e-6 (language)
- **Batch Size**: 4 (adjust for your GPU)
- **Epochs**: 10
- **Weight Decay**: 0.1
- **Gradient Clipping**: 0.1
- **Resolution**: 1008

The training setup is now complete and ready to use!